# Evaluación de resultados

En este notebook se muestran técnicas para la evaluación de los resultados de una predicción con un algoritmo de Machine Learning.

## Conjunto de datos

### Descripción
NSL-KDD es un conjunto de datos propuesto para resolver algunos de los problemas inherentes del conjunto de datos KDD'99. Aunque esta nueva versión todavía puede sufrir algunos problemas, sigue siendo un benchmark efectivo para comparar métodos de detección de intrusiones.

### Ficheros de datos utilizados
* **KDDTrain+.csv**: Conjunto de entrenamiento completo NSL-KDD en formato CSV
* **KDDTest+.csv**: Conjunto de pruebas completo NSL-KDD en formato CSV
* **Field Names.csv**: Nombres de los atributos del conjunto de datos

### Referencias
_M. Tavallaee, E. Bagheri, W. Lu, and A. Ghorbani, "A Detailed Analysis of the KDD CUP 99 Data Set," Submitted to Second IEEE Symposium on Computational Intelligence for Security and Defense Applications (CISDA), 2009._

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

## Funciones auxiliares

In [ ]:
# Nombres de columnas del dataset NSL-KDD
COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class', 'difficulty_level'
]

# Tipos de ataque considerados normales
NORMAL_LABEL = 'normal'


def load_kdd_dataset(data_path):
    """Lectura del conjunto de datos NSL-KDD desde CSV."""
    df = pd.read_csv(data_path, header=None, names=COLUMN_NAMES)
    # Eliminar la columna de nivel de dificultad (no es una característica)
    df.drop('difficulty_level', axis=1, inplace=True)
    # Mapear etiquetas a clasificación binaria: normal vs anomaly
    df['class'] = df['class'].apply(lambda x: NORMAL_LABEL if x == NORMAL_LABEL else 'anomaly')
    # Asegurar tipos numéricos en columnas continuas
    for col in df.select_dtypes(exclude=['object', 'str']).columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

In [ ]:
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

In [ ]:
# Construcción de un pipeline para los atributos numéricos
num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('rbst_scaler', RobustScaler()),
    ])

In [ ]:
# Transformador para codificar únicamente las columnas categóricas y devolver un DataFrame
class CustomOneHotEncoder(BaseEstimator, TransformerMixin):

    def __init__(self):
        self._oh = OneHotEncoder()
        self._columns = None

    def fit(self, X, y=None):
        X_cat = X.select_dtypes(include=['object', 'str'])
        self._columns = pd.get_dummies(X_cat).columns
        self._oh.fit(X_cat)
        return self

    def transform(self, X, y=None):
        X_copy = X.copy()
        X_cat = X_copy.select_dtypes(include=['object', 'str'])
        X_cat_oh = self._oh.transform(X_cat)
        X_cat_oh = pd.DataFrame(X_cat_oh.toarray(),
                                columns=self._columns,
                                index=X_copy.index)
        X_copy.drop(list(X_cat), axis=1, inplace=True)
        return X_copy.join(X_cat_oh)

In [ ]:
# Transformador que prepara todo el conjunto de datos llamando pipelines y transformadores personalizados
class DataFramePreparer(BaseEstimator, TransformerMixin):

    def __init__(self):
        self._full_pipeline = None
        self._columns = None

    def fit(self, X, y=None):
        num_attribs = list(X.select_dtypes(exclude=['object', 'str']))
        cat_attribs = list(X.select_dtypes(include=['object', 'str']))
        self._full_pipeline = ColumnTransformer([
                ("num", num_pipeline, num_attribs),
                ("cat", CustomOneHotEncoder(), cat_attribs),
        ])
        self._full_pipeline.fit(X)
        self._columns = pd.get_dummies(X).columns
        return self

    def transform(self, X, y=None):
        X_copy = X.copy()
        X_prep = self._full_pipeline.transform(X_copy)
        return pd.DataFrame(X_prep,
                            columns=self._columns,
                            index=X_copy.index)

## Lectura del conjunto de datos

In [ ]:
df = load_kdd_dataset("NSL_KDD-master/KDDTrain+.csv")

In [ ]:
df.head(10)

## División del conjunto de datos

In [ ]:
# División del conjunto en los diferentes subconjuntos
train_set, val_set, test_set = train_val_test_split(df)

In [ ]:
print("Longitud del Training Set:", len(train_set))
print("Longitud del Validation Set:", len(val_set))
print("Longitud del Test Set:", len(test_set))

Para cada uno de los subconjuntos, separamos las etiquetas de las características de entrada.

In [ ]:
# Conjunto de datos general
X_df = df.drop("class", axis=1)
y_df = df["class"].copy()

In [ ]:
# Conjunto de datos de entrenamiento
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

In [ ]:
# Conjunto de datos de validación
X_val = val_set.drop("class", axis=1)
y_val = val_set["class"].copy()

In [ ]:
# Conjunto de datos de pruebas
X_test = test_set.drop("class", axis=1)
y_test = test_set["class"].copy()

## Preparación del conjunto de datos

In [ ]:
# Instanciamos nuestro transformador personalizado
data_preparer = DataFramePreparer()

In [ ]:
# Hacemos el fit con el conjunto de datos general para que adquiera todos los valores posibles
data_preparer.fit(X_df)

In [ ]:
# Transformamos el subconjunto de datos de entrenamiento
X_train_prep = data_preparer.transform(X_train)

In [ ]:
X_train.head(5)

In [ ]:
X_train_prep.head(5)

In [ ]:
# Transformamos el subconjunto de datos de validación
X_val_prep = data_preparer.transform(X_val)

## Entrenamiento de un algoritmo de Regresión Logística

La instanciación de un algoritmo de Machine Learning utilizando Sklearn se realiza utilizando los métodos expuestos por la API de sklearn. En este caso vamos a pedirle al modelo que se entrene durante más tiempo, concretamente 1.000 iteraciones.

In [ ]:
# Entrenamos un algoritmo basado en regresión logística
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(solver="newton-cg", max_iter=1000)
clf.fit(X_train_prep, y_train)

## Predicción de nuevos ejemplos

Realizamos una predicción con el modelo generado anteriormente tras el entrenamiento del algoritmo de Regresión Logística. Utilizamos el subconjunto de validación.

In [ ]:
y_pred = clf.predict(X_val_prep)

## 1. Matriz de Confusión

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_val, y_pred)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(clf, X_val_prep, y_val, values_format='d')

## 2. Métricas derivadas de la matriz de confusión

### Precisión

In [ ]:
from sklearn.metrics import precision_score

print("Precisión:", precision_score(y_val, y_pred, pos_label='anomaly'))

### Recall

In [ ]:
from sklearn.metrics import recall_score

print("Recall:", recall_score(y_val, y_pred, pos_label='anomaly'))

### F1 Score

In [ ]:
from sklearn.metrics import f1_score

print("F1 score:", f1_score(y_val, y_pred, pos_label='anomaly'))

## 3. Curvas ROC y PR

### Curva ROC

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_estimator(clf, X_val_prep, y_val, pos_label='anomaly')

### Curva PR

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_estimator(clf, X_val_prep, y_val, pos_label='anomaly')

## 4. Evaluación del modelo con el conjunto de datos de pruebas

In [ ]:
# Transformamos el subconjunto de datos de pruebas
X_test_prep = data_preparer.transform(X_test)

In [ ]:
y_pred = clf.predict(X_test_prep)

In [ ]:
ConfusionMatrixDisplay.from_estimator(clf, X_test_prep, y_test, values_format='d')

In [ ]:
print("F1 score:", f1_score(y_test, y_pred, pos_label='anomaly'))